# FourierSurface

A closed toroidal surface as a Fourier series: a stellarator's plasma boundary, or a first
wall around it. The layout is VMEC's and ConStellaration's `SurfaceRZFourier`:
$R = \sum r^c_{mn} \cos(m\theta - N_\mathrm{fp} n\varphi)$ and
$Z = \sum z^s_{mn} \sin(m\theta - N_\mathrm{fp} n\varphi)$, with $m = 0 \dots M$ along the
first axis of the arrays, $n = -N \dots N$ along the second and $\varphi$ the cylindrical angle.

Membership and the search for segment hits use a triangulation of the surface (a
`TriangleMesh`); the hits are then moved onto the exact surface by Newton steps.

In [ ]:
import numpy as np

from simgeometry import FourierSurface

# a circular torus: R = 3 + 0.5 cos(theta), Z = 0.5 sin(theta)
r_cos = np.array([[3.0], [0.5]])
z_sin = np.array([[0.0], [0.5]])
torus = FourierSurface(r_cos, z_sin, n_field_periods=1)
print(
    "volume:", round(torus.volume(), 6), "exact:", round(2 * np.pi**2 * 3.0 * 0.25, 6)
)
print("inside at R = 3:", torus.contains((3.0, 0.0, 0.0)))
print("triangulation error:", round(torus.triangulation_error, 5))
assert np.isclose(torus.volume(), 2 * np.pi**2 * 3.0 * 0.25)

## Hits and offsets

`first_hit` takes segments as `(..., 3)` arrays and gives the hit, its distance, point,
outward normal and the surface's angles there. `offset` moves the surface along its normal:
for a circular torus by exactly the distance in the minor radius.

In [ ]:
hits = torus.first_hit([[0.0, 0.0, 0.0]], [[5.0, 0.0, 0.0]])
print("distance:", hits.distance[0], "normal:", hits.normal[0])
wall = torus.offset(0.2)
print("modes:", wall.n_poloidal, wall.n_toroidal, "minor radius:", wall.r_cos[1, 0])
print("misfit:", wall.offset_residual)
assert np.isclose(hits.distance[0], 2.5) and np.isclose(wall.r_cos[1, 0], 0.7)

## A stellarator boundary

`FourierSurface.from_boundary` reads ConStellaration's `boundary.json` (a dict, its JSON or a
`SurfaceRZFourier`). `simgeometry.data.w7x_boundary()` is W7-X's, from DESC's example
equilibrium; its sections over half a field period:

In [ ]:
import matplotlib.pyplot as plt

from simgeometry.data import w7x_boundary_data

boundary = FourierSurface.from_boundary(w7x_boundary_data())
theta = np.linspace(0, 2 * np.pi, 300)
fig, ax = plt.subplots(figsize=(5, 4))
for fraction in (0.0, 0.125, 0.25, 0.375, 0.5):
    phi = fraction * 2 * np.pi / boundary.nfp
    ax.plot(*boundary.rz(theta, phi), label=f"{np.degrees(phi):.0f}°")
ax.set_aspect("equal")
ax.set_xlabel("R [m]")
ax.set_ylabel("Z [m]")
ax.legend(title="$\\varphi$", fontsize=8)
ax.set_title("W7-X boundary (DESC example) over half a field period")
plt.show()